# Assignment NLP - 5 (Token Classification: POS Tagging & Chunking)
## Fine-Tuning BERT for POS Tagging & Chunking

### Name: Siddharth
### Intern ID: [Your Intern ID]

---
## Task 1: Dataset Selection
- **Dataset Name:** `conll2003`
- **Description:** The CoNLL-2003 dataset contains text from Reuters news stories. It includes tags for POS (Part-of-Speech), syntactic chunks, and Named Entities (NER). For this task, we will focus on **POS tags** and **Chunk tags**.
- **Label Categories (POS):** NNP, NN, IN, DT, JJ, etc.
- **Label Categories (Chunking):** B-NP, I-NP, B-VP, I-VP, B-PP, etc.



In [15]:
# Install required libraries
!pip install -q transformers datasets seqeval evaluate accelerate


## Task 2: Data Preprocessing
We will load the `conll2003` dataset and preprocess it. We will use DistilBERT for faster training. Note that DistilBERT uses subword tokenization, so we must align the labels with the tokens. Subword tokens that don't match the start of a word will be assigned a label of `-100` so they are ignored in the loss calculation.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load Dataset
dataset = load_dataset("conll2003")
print(dataset)

# Display available features
pos_features = dataset["train"].features["pos_tags"]
chunk_features = dataset["train"].features["chunk_tags"]
print("POS Labels:", pos_features.feature.names)
print("Chunk Labels:", chunk_features.feature.names)

# We will focus on POS Tagging for the main training pipeline.
label_list = pos_features.feature.names
num_labels = len(label_list)

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}


In [ ]:
# Tokenization and Label Alignment
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples, label_column="pos_tags"):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens mapped to None are labeled as -100
            if word_idx is None:
                label_ids.append(-100)
            # Assign the label to the first token of the word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens of the same word, assign -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the preprocessing function to the dataset
tokenized_datasets = dataset.map(lambda x: tokenize_and_align_labels(x, "pos_tags"), batched=True)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Task 3: Model Setup
We set up `AutoModelForTokenClassification` for POS tagging using DistilBERT.


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


## Task 4 & 5: Training and Evaluation
We will use the Hugging Face Trainer API and the `seqeval` metric to evaluate Precision, Recall, and F1 Score.


In [5]:
import numpy as np
import evaluate

# Load seqeval metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./pos_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(5000)), # Subset for faster training in Colab/Local
    eval_dataset=tokenized_datasets["validation"].select(range(1000)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start Model Training
trainer.train()


## Task 6: Inference
Let's load the trained model and use a pipeline for token classification.


In [ ]:
from transformers import pipeline

# We can specify the model folder if saved, but we can also use the in-memory 'model' and 'tokenizer'
token_classifier = pipeline(
    "token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple"
)

text = "John works at Google in California."
outputs = token_classifier(text)

import pandas as pd
pd.options.display.max_columns = None
print("\n--- POS TAGGING INFERENCE ---")
display(pd.DataFrame(outputs))


## Task 7: Comparison (POS Tagging vs Chunking)
- **POS Tagging (Grammar-level tagging):** Very granular. Classifies each token into parts of speech like Noun, Verb, Adjective, etc. Generally considered **Easy** locally since it relies heavily on surrounding small word window context.
- **Chunking (Phrase-level grouping):** Medium difficulty. Groups adjacent tokens into structures like Noun Phrases (NP) or Verb Phrases (VP). Requires understanding the syntactic boundary of phrases rather than individual word roles.

*Note: For chunking, the exact same pipeline could be applied simply by passing `label_column="chunk_tags"` above and updating the label_list.*


## Task 8: Report / Blog

**Summary & Learnings**
During this assignment, I fine-tuned a DistilBERT model for token classification to perform POS tagging.

**Key Learnings:**
1. **Token alignment:** Subword tokenization complicates sequence labeling. Setting subsequent subword label IDs to -100 ensures the loss function computes correctly for the true tokens.
2. **Seqeval Metric:** Standard classification metrics like accuracy are bad for token classification because most tokens are simple or 'O' classes. Seqeval evaluates based on actual entity span precision, recall, and F1.
3. **Pipeline flexibility:** The exact same pipeline can be applied seamlessly to POS tagging, Chunking, or NER just by modifying the training labels.

**Challenges Faced:**
Memory constraints were notable; I downsampled the dataset slightly to iterate faster, given constraints on a single consumer GPU or Colab environment. Matching the exact formatting for subword labels was initially challenging.

**LinkedIn Post Text Example:**
```text
I just completed an interesting project on fine-tuning Transformer models like DistilBERT for Token Classification! 🚀

Summary: I successfully trained a DistilBERT model on the CoNLL-2003 dataset to perform Part-of-Speech (POS) Tagging and analyzed its conceptual difference against Phrase Chunking.

Key Learnings:
- Handled label alignment challenges created by subword tokenization (WordPiece).
- Implemented and evaluated the model using the rigorous 'seqeval' metric for token sequences.
- Gained hands-on experience utilizing Hugging Face's Trainer API for seamless fine-tuning.

A big thank you to @Innomatics Research Labs for the guidance!

#NLP #AI #DataScience #MachineLearning #Transformers #DeepLearning
```
